# EP model demo

Thin demo: reads `plays_scored.parquet` (produced by `ffep score`) and visualizes the trained EP model's output. No model fitting happens here -- run `ffep train --model ep` then `ffep score` to (re)produce the scored data. See `docs/pipeline.md` for the CLI reference and MLflow tracking details.

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path

scored = pl.read_parquet(Path('../data/processed') / 'plays_scored.parquet')
print(f'plays_scored: {scored.height} rows')

## Expected points (EP) by field position, per down

In [ ]:
ep_by_down = (
    scored.filter(pl.col('ep').is_not_null(), pl.col('down').is_in([1, 2, 3, 4]))
    .sort(['down', 'yardline_50'])
)

fig, ax = plt.subplots(figsize=(8, 5))
for down, group in ep_by_down.group_by('down', maintain_order=True):
    ax.scatter(group['yardline_50'], group['ep'], s=8, alpha=0.4, label=f'down {down[0]}')
ax.set_xlabel('yardline_50 (yards from opponent goal line)')
ax.set_ylabel('ep (expected points)')
ax.set_title('EP by field position, per down')
ax.legend()
plt.show()

## EPA leaders by team (`posteam`)

In [ ]:
epa_leaders = (
    scored.filter(pl.col('epa').is_not_null())
    .group_by('posteam')
    .agg(
        total_epa=pl.col('epa').sum(),
        mean_epa=pl.col('epa').mean(),
        n_plays=pl.len(),
    )
    .sort('total_epa', descending=True)
)
epa_leaders.head(15)